# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sanaanwar25/flyrank-ml-internship-sana/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os

REPO_DIR = "/content/flyrank-ml-internship-sana"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/sanaanwar25/flyrank-ml-internship-sana.git

os.chdir(REPO_DIR)

print("Repository:", os.getcwd())
print("Repository exists:", os.path.exists(".git"))

Repository: /content/flyrank-ml-internship-sana
Repository exists: True


# Abstract

This project investigates whether a machine-learning scoring approach can help prioritize pages for content review. The analysis uses anonymized search and content-performance data to identify pages that may deserve attention. A transparent baseline is compared with a learned model using a held-out evaluation design. The findings are observed and directional decision-support evidence, not causal proof or a prediction of Google's ranking algorithm.

## 1. Question

*The research question and the decision it supports.*

This paper asks whether a leakage-safe ML model can rank content pages by their observed likelihood of needing attention better than a simple hand-written baseline rule. The decision supported is which pages a content team should review first. The goal is decision-support and prioritization, not causal proof or prediction of Google's ranking algorithm.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

I use the provided FlyRank full-release warehouse through DuckDB and Hugging Face. The analysis uses anonymized client, content, daily performance, and 90-day query-level data.

The feature-building workflow uses a 90-day historical window. Content items are retained when they have at least 100 impressions in the previous 30-day period, providing a minimum history threshold for the analysis.

The model uses pre-outcome search-performance and query-level signals, including previous-30-day impressions, visible query count, rare-query share, anonymous-query share, and top-query share. Client names, domains, URLs, private queries, credentials, and raw exports are not used in the public report.

Future outcome information is not used as a model feature in order to reduce leakage.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

The analysis defines a declining content item as one whose impressions in the last 30 days are less than 80% of its impressions in the previous 30 days.

The model uses five features available before the outcome window: previous-30-day impressions, visible query count, rare-query share, anonymous-query share, and top-query share.

A Random Forest classifier with 200 trees is used as the first learned model. The transparent baseline always predicts the majority class in the held-out test set.

Two validation designs are compared. First, a random 75/25 train-test split is used as a simple benchmark. Second, GroupShuffleSplit is used with client_hash_id as the grouping variable so that clients represented in the training data are separated from clients in the test data.

The grouped split is used to examine whether the observed signal survives a stricter cross-client evaluation. Label-derived and future-window information are excluded from the feature set to reduce leakage.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

The 90-day feature workflow produced 136,252 content items with enough history for analysis. The decline label marks an item as declining when impressions in the last 30 days are less than 80% of impressions in the previous 30 days.

Using a random 75/25 train-test split, the Random Forest achieved 0.915 accuracy on the held-out test set, compared with a majority-class baseline accuracy of 0.874. For the declining class, the model achieved 0.989 recall and an F1-score of 0.953.

To test whether the result generalized across clients, the random split was replaced with a client-grouped holdout using GroupShuffleSplit on client_hash_id. The grouped test set contained 14,728 rows from 13 clients. The grouped evaluation achieved 0.933 accuracy, with a majority-class baseline of 0.867. The macro F1-score was 0.820 and the weighted F1-score was 0.924.

The grouped result is more informative about cross-client generalization than the random split because pages from the same client are kept out of both training and testing. These results are observational and directional; they do not establish causality or predict Google's ranking algorithm.

## 5. Limitations

*What this work cannot claim.*

This analysis cannot prove that any feature causes a change in search performance. It cannot explain or predict Google's ranking algorithm, and it cannot guarantee that a recommended page will improve after editing. The results are limited to the observed anonymized dataset, its time windows, its label definition, and the validation design. The ranked queue should therefore be treated as directional decision-support for review prioritization.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

The model output should be treated as a reviewer aid rather than an automatic publishing decision.

The recommended workflow is to review higher-priority content first and combine the model's score with editorial judgment. A high model score indicates that an item resembles content associated with the observed decline label in this dataset; it does not guarantee that a refresh will improve performance.

Reviewers should check the underlying content and search-performance context before taking action. Items should not be automatically rewritten, removed, or published based only on the model score.

In [ ]:
import os
import pandas as pd

path = "outputs/refresh_queue_sample.csv"

if os.path.exists(path):
    queue = pd.read_csv(path)
    print("Loaded:", path)
    print("Rows:", len(queue))
    display(queue.head(20))
else:
    print("Ranked queue not found.")

Loaded: outputs/refresh_queue_sample.csv
Rows: 200


,final_rank,content_id,client_id,final_refresh_score,best_model_name,best_model_probability,baseline_refresh_score,confidence,suggested_action,final_reason_codes,...,word_count,trend_direction,competition_level,content_type,main_intent,age_tier,freshness_tier,word_count_tier,impression_tier,position_tier
0,1,content_1f080331fa2b,client_3fdba35f04,81.636697,random_forest,0.782079,0.844481,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|low...,...,1404.0,down,MEDIUM,keyword article,informational,91-180,91-180,1000-2000,good,page_1
1,2,content_6aa43079fb0c,client_3fdba35f04,81.447656,random_forest,0.788105,0.825477,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,...,1457.0,down,LOW,keyword article,informational,91-180,91-180,1000-2000,good,page_1
2,3,content_d6570c51c9bd,client_3fdba35f04,81.430346,random_forest,0.847372,0.695884,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,...,1362.0,down,MEDIUM,keyword article,informational,91-180,91-180,1000-2000,moderate,striking
3,4,content_72e800a9c214,client_3fdba35f04,81.034960,random_forest,0.774371,0.842545,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,...,1371.0,down,MEDIUM,keyword article,commercial,91-180,91-180,1000-2000,good,page_1
4,5,content_e04eb9549989,client_3fdba35f04,80.873188,random_forest,0.814805,0.749468,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,...,1408.0,down,LOW,keyword article,informational,91-180,91-180,1000-2000,good,page_1
5,6,content_b69288c5e701,client_3fdba35f04,80.754770,random_forest,0.795713,0.787358,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,...,1370.0,down,LOW,keyword article,informational,91-180,91-180,1000-2000,good,page_1
6,7,content_9b6df29f7889,client_3fdba35f04,80.632923,random_forest,0.846245,0.673530,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,...,1415.0,down,MEDIUM,keyword article,commercial,91-180,91-180,1000-2000,moderate,page_1
7,8,content_bb6ebb5ec8c8,client_3fdba35f04,80.371236,random_forest,0.834638,0.690665,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,...,1381.0,down,LOW,keyword article,informational,91-180,91-180,1000-2000,moderate,striking
8,9,content_4d76cdb3387b,client_3fdba35f04,80.362748,random_forest,0.843092,0.671993,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,...,1554.0,down,HIGH,keyword article,commercial,91-180,91-180,1000-2000,moderate,top_3
9,10,content_b4f35d640b1c,client_3fdba35f04,80.321757,random_forest,0.843803,0.669168,medium,refresh,declining_with_demand|model_decline_risk|visib...,...,1389.0,down,HIGH,keyword article,commercial,91-180,91-180,1000-2000,good,page_3_5


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Artifacts

The repository contains output artifacts used to document the analysis, including a ranked queue sample and a model report. These artifacts provide supporting evidence for the prioritization workflow and make the generated results easier to inspect.

In [ ]:
import os

print("Checking available output artifacts...")

for root, dirs, files in os.walk("."):
    for name in files:
        if name.endswith((".png", ".csv", ".md", ".json")):
            path = os.path.join(root, name)
            if "outputs" in path:
                print(path)

Checking available output artifacts...
./outputs/refresh_queue_sample.csv
./outputs/model_report.md


## 8. Acknowledgments & Data Credit

This project was completed as part of the FlyRank ML internship workflow. The analysis uses the provided anonymized, public-safe dataset and follows the project guidance for responsible use of the data.

Data and project context are credited to FlyRank.

[FlyRank](https://flyrank.ai)

## 5-Minute Demo Outline

**0:00–0:30 — Problem:** Explain the goal of prioritizing pages for content review.

**0:30–1:30 — Data:** Explain the anonymized page-level dataset and the leakage-safe feature window.

**1:30–2:30 — Method:** Explain the model, baseline, label, and client-grouped holdout design.

**2:30–3:30 — Results:** Show the ranked queue, confidence groups, recommended actions, and important features.

**3:30–4:30 — Recommendations:** Explain how reviewers should use the queue to prioritize manual review.

**4:30–5:00 — Limitations:** Explain that the results are directional decision-support evidence, not causal proof or a prediction of Google's ranking algorithm.

## Social-Post Cut

Built a leakage-safe ML workflow to prioritize content pages for review using anonymized search-performance data. The project compares a learned model with a transparent baseline and produces a ranked review queue with confidence levels and recommended actions. The results are intended as practical decision-support evidence, not a prediction of Google's ranking algorithm.

## Employer-Facing Summary

I built a leakage-aware ML prioritization workflow that uses anonymized search-performance data to rank pages for content review. I evaluated the workflow using a client-grouped holdout design and produced a practical review queue with confidence levels, recommended actions, and feature-importance information. The resulting system demonstrates an end-to-end approach to turning historical data into transparent, actionable decision-support while clearly communicating its limitations.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
